# Assignment #03

by Patrick Donnelly & Burke Havranek

EECE 5644: Introduction to Machine Learning and Pattern Recognition

Northeastern University College of Engineering

Summer 2026, Session B

## Scenario

### **1. Company Background**

WheelsBazaar is an online used-car marketplace operating across India (similar to Cars24, CarDekho, or
Spinny). Sellers list their vehicles on our platform, and buyers browse, compare, and purchase. We currently
handle roughly 5,000 listings per month across metros and tier-2 cities.

### **2. Business Problem**
Today, sellers set their own asking price, and our operations team manually reviews listings that "look off." This
causes three measurable problems:
1. **Overpriced listings sit unsold.** Cars priced more than ~15% above market value take 3x longer to sell,
clogging inventory and hurting buyer trust in the platform.
2. **Underpriced listings cost sellers money** — and generate complaints when sellers later discover they
sold below market. This damages retention.
3. **Manual review doesn't scale.** Our pricing analysts can review ~40 listings a day. At 5,000
listings/month we review less than 25% of inventory.
We also plan to launch an instant-buy program (WheelsBazaar Direct) where the company purchases cars
directly from sellers. For this, we need an auto

## Part 1: Data Retrieval and Sanitization

In [1]:
#!/bin/python3
# --Beginning of Code--
import sys, subprocess
def pipq(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs])

pipq("scikit-learn", "pandas", "numpy", "matplotlib", "seaborn", "pandas")

import numpy as np
import pandas as pd
pd.set_option("display.max_columns", 80)
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

df = pd.read_csv("car_details_v4.csv")
df

,Make,Model,Price,Year,Kilometer,Fuel Type,Transmission,Location,Color,Owner,Seller Type,Engine,Max Power,Max Torque,Drivetrain,Length,Width,Height,Seating Capacity,Fuel Tank Capacity
0,Honda,Amaze 1.2 VX i-VTEC,505000,2017,87150,Petrol,Manual,Pune,Grey,First,Corporate,1198 cc,87 bhp @ 6000 rpm,109 Nm @ 4500 rpm,FWD,3990.0,1680.0,1505.0,5.0,35.0
1,Maruti Suzuki,Swift DZire VDI,450000,2014,75000,Diesel,Manual,Ludhiana,White,Second,Individual,1248 cc,74 bhp @ 4000 rpm,190 Nm @ 2000 rpm,FWD,3995.0,1695.0,1555.0,5.0,42.0
2,Hyundai,i10 Magna 1.2 Kappa2,220000,2011,67000,Petrol,Manual,Lucknow,Maroon,First,Individual,1197 cc,79 bhp @ 6000 rpm,112.7619 Nm @ 4000 rpm,FWD,3585.0,1595.0,1550.0,5.0,35.0
3,Toyota,Glanza G,799000,2019,37500,Petrol,Manual,Mangalore,Red,First,Individual,1197 cc,82 bhp @ 6000 rpm,113 Nm @ 4200 rpm,FWD,3995.0,1745.0,1510.0,5.0,37.0
4,Toyota,Innova 2.4 VX 7 STR [2016-2020],1950000,2018,69000,Diesel,Manual,Mumbai,Grey,First,Individual,2393 cc,148 bhp @ 3400 rpm,343 Nm @ 1400 rpm,RWD,4735.0,1830.0,1795.0,7.0,55.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2054,Mahindra,XUV500 W8 [2015-2017],850000,2016,90300,Diesel,Manual,Surat,White,First,Individual,2179 cc,138 bhp @ 3750 rpm,330 Nm @ 1600 rpm,FWD,4585.0,1890.0,1785.0,7.0,70.0
2055,Hyundai,Eon D-Lite +,275000,2014,83000,Petrol,Manual,Ahmedabad,White,Second,Individual,814 cc,55 bhp @ 5500 rpm,75 Nm @ 4000 rpm,FWD,3495.0,1550.0,1500.0,5.0,32.0
2056,Ford,Figo Duratec Petrol ZXI 1.2,240000,2013,73000,Petrol,Manual,Thane,Silver,First,Individual,1196 cc,70 bhp @ 6250 rpm,102 Nm @ 4000 rpm,FWD,3795.0,1680.0,1427.0,5.0,45.0
2057,BMW,5-Series 520d Luxury Line [2017-2019],4290000,2018,60474,Diesel,Automatic,Coimbatore,White,First,Individual,1995 cc,188 bhp @ 4000 rpm,400 Nm @ 1750 rpm,RWD,4936.0,1868.0,1479.0,5.0,65.0


Thus, we find 2059 entries with 20 individual fields. First, we examine any missing or malformed entries:

In [2]:
df.isnull().sum()

Make                    0
Model                   0
Price                   0
Year                    0
Kilometer               0
Fuel Type               0
Transmission            0
Location                0
Color                   0
Owner                   0
Seller Type             0
Engine                 80
Max Power              80
Max Torque             80
Drivetrain            136
Length                 64
Width                  64
Height                 64
Seating Capacity       64
Fuel Tank Capacity    113
dtype: int64

In [3]:
df.isna().sum()

Make                    0
Model                   0
Price                   0
Year                    0
Kilometer               0
Fuel Type               0
Transmission            0
Location                0
Color                   0
Owner                   0
Seller Type             0
Engine                 80
Max Power              80
Max Torque             80
Drivetrain            136
Length                 64
Width                  64
Height                 64
Seating Capacity       64
Fuel Tank Capacity    113
dtype: int64

We find several across several fields with missing entries. To determine whether they are correlated, let us examine first the most deficient field:

In [4]:
df[df["Drivetrain"].isnull()].head(10)

,Make,Model,Price,Year,Kilometer,Fuel Type,Transmission,Location,Color,Owner,Seller Type,Engine,Max Power,Max Torque,Drivetrain,Length,Width,Height,Seating Capacity,Fuel Tank Capacity
33,Honda,CR-V 2.4 AT,860000,2013,67000,Petrol,Automatic,Mumbai,Brown,First,Individual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69,Audi,A4 2.0 TDI (143 bhp),1250000,2012,50000,Diesel,Automatic,Mumbai,White,First,Individual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
94,Mercedes-Benz,GLC 220 d Sport,3900000,2018,83400,Diesel,Automatic,Hyderabad,White,Second,Individual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
108,Honda,Brio S MT,229000,2013,38175,Petrol,Manual,Kolkata,Blue,First,Individual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
117,Hyundai,Santro GL LPG,120000,2009,48500,LPG,Manual,Lucknow,Grey,Fourth,Individual,1086 cc,63@5500,89@3000,NaN,3565.0,1525.0,1590.0,5.0,35.0
138,Maruti Suzuki,Ritz VXI BS-IV,210000,2011,58888,Petrol,Manual,Delhi,Maroon,Second,Individual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
147,Mercedes-Benz,GLC 220 d Sport,4400000,2018,47009,Diesel,Automatic,Pune,Blue,Second,Individual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
162,MG,Hector Sharp 1.5 DCT Petrol [2019-2020],1729000,2020,27000,Petrol,Automatic,Panvel,Silver,First,Individual,1451 cc,141 bhp @ 5000 rpm,250 Nm @ 1600 rpm,NaN,4655.0,1835.0,1760.0,5.0,60.0
177,Maruti Suzuki,Wagon R VXi,350000,2013,45000,Petrol,Manual,Pune,White,First,Individual,998 cc,68@6200,90@3500,NaN,3539.0,1495.0,1700.0,5.0,35.0
192,Toyota,Innova 2.5 G4 7 STR,645000,2013,76000,Diesel,Manual,Kanpur,Grey,Second,Individual,2494 cc,102@5600,200@1400,NaN,4555.0,1770.0,1755.0,7.0,55.0


From this, we find that the spread of invalid entries tend to correlate quite strongly with each other, but not with any other fields in particular. We are unsure as to the relative importance of each field in determining price, however. Since these incomplete entries compose a rather small percentage of the total data set (~9%), and due to a lack of any defensible default value for the categorical value of `Drivetrain`, we have therefore chosen to drop incomplete entries.

In [5]:
df.dropna(inplace=True)
df

,Make,Model,Price,Year,Kilometer,Fuel Type,Transmission,Location,Color,Owner,Seller Type,Engine,Max Power,Max Torque,Drivetrain,Length,Width,Height,Seating Capacity,Fuel Tank Capacity
0,Honda,Amaze 1.2 VX i-VTEC,505000,2017,87150,Petrol,Manual,Pune,Grey,First,Corporate,1198 cc,87 bhp @ 6000 rpm,109 Nm @ 4500 rpm,FWD,3990.0,1680.0,1505.0,5.0,35.0
1,Maruti Suzuki,Swift DZire VDI,450000,2014,75000,Diesel,Manual,Ludhiana,White,Second,Individual,1248 cc,74 bhp @ 4000 rpm,190 Nm @ 2000 rpm,FWD,3995.0,1695.0,1555.0,5.0,42.0
2,Hyundai,i10 Magna 1.2 Kappa2,220000,2011,67000,Petrol,Manual,Lucknow,Maroon,First,Individual,1197 cc,79 bhp @ 6000 rpm,112.7619 Nm @ 4000 rpm,FWD,3585.0,1595.0,1550.0,5.0,35.0
3,Toyota,Glanza G,799000,2019,37500,Petrol,Manual,Mangalore,Red,First,Individual,1197 cc,82 bhp @ 6000 rpm,113 Nm @ 4200 rpm,FWD,3995.0,1745.0,1510.0,5.0,37.0
4,Toyota,Innova 2.4 VX 7 STR [2016-2020],1950000,2018,69000,Diesel,Manual,Mumbai,Grey,First,Individual,2393 cc,148 bhp @ 3400 rpm,343 Nm @ 1400 rpm,RWD,4735.0,1830.0,1795.0,7.0,55.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2053,Maruti Suzuki,Ritz Vxi (ABS) BS-IV,245000,2014,79000,Petrol,Manual,Faridabad,White,Second,Individual,1197 cc,85 bhp @ 6000 rpm,113 Nm @ 4500 rpm,FWD,3775.0,1680.0,1620.0,5.0,43.0
2054,Mahindra,XUV500 W8 [2015-2017],850000,2016,90300,Diesel,Manual,Surat,White,First,Individual,2179 cc,138 bhp @ 3750 rpm,330 Nm @ 1600 rpm,FWD,4585.0,1890.0,1785.0,7.0,70.0
2055,Hyundai,Eon D-Lite +,275000,2014,83000,Petrol,Manual,Ahmedabad,White,Second,Individual,814 cc,55 bhp @ 5500 rpm,75 Nm @ 4000 rpm,FWD,3495.0,1550.0,1500.0,5.0,32.0
2056,Ford,Figo Duratec Petrol ZXI 1.2,240000,2013,73000,Petrol,Manual,Thane,Silver,First,Individual,1196 cc,70 bhp @ 6250 rpm,102 Nm @ 4000 rpm,FWD,3795.0,1680.0,1427.0,5.0,45.0


Let us now consider the format of each entry for each field. For such entries as `Make` and `Model`, these fields are purely categorical, and cannot be meaningfully reduced further other than through encoding, as discussed below. However, for such fields as `Engine`, `Max Power`, and `Max Torque`, these fields carry continuous numerical information in addition to units, and may therefore uniformly sanitized as follows:

In [6]:
# Given a string, extracts the first floating-point number
get_first_number = lambda x: float(re.search(r'^\d+\.?\d*', x).group())

df["Engine"] = df["Engine"].apply(get_first_number)
df["Max Power"] = df["Max Power"].apply(get_first_number)
df["Max Torque"] = df["Max Torque"].apply(get_first_number)

df

,Make,Model,Price,Year,Kilometer,Fuel Type,Transmission,Location,Color,Owner,Seller Type,Engine,Max Power,Max Torque,Drivetrain,Length,Width,Height,Seating Capacity,Fuel Tank Capacity
0,Honda,Amaze 1.2 VX i-VTEC,505000,2017,87150,Petrol,Manual,Pune,Grey,First,Corporate,1198.0,87.0,109.0000,FWD,3990.0,1680.0,1505.0,5.0,35.0
1,Maruti Suzuki,Swift DZire VDI,450000,2014,75000,Diesel,Manual,Ludhiana,White,Second,Individual,1248.0,74.0,190.0000,FWD,3995.0,1695.0,1555.0,5.0,42.0
2,Hyundai,i10 Magna 1.2 Kappa2,220000,2011,67000,Petrol,Manual,Lucknow,Maroon,First,Individual,1197.0,79.0,112.7619,FWD,3585.0,1595.0,1550.0,5.0,35.0
3,Toyota,Glanza G,799000,2019,37500,Petrol,Manual,Mangalore,Red,First,Individual,1197.0,82.0,113.0000,FWD,3995.0,1745.0,1510.0,5.0,37.0
4,Toyota,Innova 2.4 VX 7 STR [2016-2020],1950000,2018,69000,Diesel,Manual,Mumbai,Grey,First,Individual,2393.0,148.0,343.0000,RWD,4735.0,1830.0,1795.0,7.0,55.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2053,Maruti Suzuki,Ritz Vxi (ABS) BS-IV,245000,2014,79000,Petrol,Manual,Faridabad,White,Second,Individual,1197.0,85.0,113.0000,FWD,3775.0,1680.0,1620.0,5.0,43.0
2054,Mahindra,XUV500 W8 [2015-2017],850000,2016,90300,Diesel,Manual,Surat,White,First,Individual,2179.0,138.0,330.0000,FWD,4585.0,1890.0,1785.0,7.0,70.0
2055,Hyundai,Eon D-Lite +,275000,2014,83000,Petrol,Manual,Ahmedabad,White,Second,Individual,814.0,55.0,75.0000,FWD,3495.0,1550.0,1500.0,5.0,32.0
2056,Ford,Figo Duratec Petrol ZXI 1.2,240000,2013,73000,Petrol,Manual,Thane,Silver,First,Individual,1196.0,70.0,102.0000,FWD,3795.0,1680.0,1427.0,5.0,45.0


Next, we deal with the categorical data, beginning with the number of unique entries:

In [7]:
df.nunique()

Make                   32
Model                 955
Price                 586
Year                   18
Kilometer             790
Fuel Type               7
Transmission            2
Location               75
Color                  16
Owner                   4
Seller Type             3
Engine                104
Max Power             156
Max Torque            135
Drivetrain              3
Length                234
Width                 163
Height                189
Seating Capacity        6
Fuel Tank Capacity     55
dtype: int64

As expected, the categories of `Fuel Type`, `Transmission`, `Owner`, `Seller Type`, and `Drivetrain` have few entries. For `Color`, `Make`, and especially `Location` and `Model` have significantly more unique entries, the latter most comprising almost half unique entries. It is at this point that we must determine what features are significant to the cost of a vehicle.

While the `Model` of a car certainly affects its price, the sheer number of unique entries relative to the number of total entries makes any model training unlikely to converge upon a pattern, having only on average about two vehicles per unique model. We must therefore drop the `Model` field so as to not introduce undue noise to the model training.

The `Make` of a car, however, is significant to its price, with luxury brands commanding on average higher prices than budget brands. As such, this field must be kept.

The `Location` and `Color` of a car are both interesting pieces of categorical data, as it is unclear whether they might affect the car's final price. For now, we keep them, but may later drop them during model tuning.

In [8]:
df.drop("Model", axis=1, inplace=True)
df

,Make,Price,Year,Kilometer,Fuel Type,Transmission,Location,Color,Owner,Seller Type,Engine,Max Power,Max Torque,Drivetrain,Length,Width,Height,Seating Capacity,Fuel Tank Capacity
0,Honda,505000,2017,87150,Petrol,Manual,Pune,Grey,First,Corporate,1198.0,87.0,109.0000,FWD,3990.0,1680.0,1505.0,5.0,35.0
1,Maruti Suzuki,450000,2014,75000,Diesel,Manual,Ludhiana,White,Second,Individual,1248.0,74.0,190.0000,FWD,3995.0,1695.0,1555.0,5.0,42.0
2,Hyundai,220000,2011,67000,Petrol,Manual,Lucknow,Maroon,First,Individual,1197.0,79.0,112.7619,FWD,3585.0,1595.0,1550.0,5.0,35.0
3,Toyota,799000,2019,37500,Petrol,Manual,Mangalore,Red,First,Individual,1197.0,82.0,113.0000,FWD,3995.0,1745.0,1510.0,5.0,37.0
4,Toyota,1950000,2018,69000,Diesel,Manual,Mumbai,Grey,First,Individual,2393.0,148.0,343.0000,RWD,4735.0,1830.0,1795.0,7.0,55.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2053,Maruti Suzuki,245000,2014,79000,Petrol,Manual,Faridabad,White,Second,Individual,1197.0,85.0,113.0000,FWD,3775.0,1680.0,1620.0,5.0,43.0
2054,Mahindra,850000,2016,90300,Diesel,Manual,Surat,White,First,Individual,2179.0,138.0,330.0000,FWD,4585.0,1890.0,1785.0,7.0,70.0
2055,Hyundai,275000,2014,83000,Petrol,Manual,Ahmedabad,White,Second,Individual,814.0,55.0,75.0000,FWD,3495.0,1550.0,1500.0,5.0,32.0
2056,Ford,240000,2013,73000,Petrol,Manual,Thane,Silver,First,Individual,1196.0,70.0,102.0000,FWD,3795.0,1680.0,1427.0,5.0,45.0


Let us finally verify the aforementioned categorical data:

In [9]:
print(df["Make"].unique(),end='\n\n')
print(df["Fuel Type"].unique(),end='\n\n')
print(df["Transmission"].unique(),end='\n\n')
print(df["Location"].unique(),end='\n\n')
print(df["Color"].unique(),end='\n\n')
print(df["Owner"].unique(),end='\n\n')
print(df["Seller Type"].unique(),end='\n\n')
print(df["Drivetrain"].unique(),end='\n\n')

['Honda' 'Maruti Suzuki' 'Hyundai' 'Toyota' 'BMW' 'Skoda' 'Nissan'
 'Renault' 'Tata' 'Volkswagen' 'Ford' 'Mercedes-Benz' 'Audi' 'Mahindra'
 'MG' 'Jeep' 'Porsche' 'Kia' 'Land Rover' 'Volvo' 'Maserati' 'Jaguar'
 'Isuzu' 'MINI' 'Ferrari' 'Mitsubishi' 'Datsun' 'Chevrolet' 'Ssangyong'
 'Fiat' 'Rolls-Royce' 'Lexus']

['Petrol' 'Diesel' 'CNG' 'CNG + CNG' 'LPG' 'Hybrid' 'Petrol + CNG']

['Manual' 'Automatic']

['Pune' 'Ludhiana' 'Lucknow' 'Mangalore' 'Mumbai' 'Coimbatore' 'Bangalore'
 'Delhi' 'Raipur' 'Kanpur' 'Patna' 'Vadodara' 'Hyderabad' 'Yamunanagar'
 'Gurgaon' 'Jaipur' 'Deoghar' 'Agra' 'Goa' 'Warangal' 'Jalandhar' 'Noida'
 'Ahmedabad' 'Mohali' 'Ghaziabad' 'Kolkata' 'Zirakpur' 'Nagpur' 'Thane'
 'Faridabad' 'Ranchi' 'Chandigarh' 'Amritsar' 'Chennai' 'Navi Mumbai'
 'Udupi' 'Jamshedpur' 'Aurangabad' 'Rudrapur' 'Nashik' 'Varanasi' 'Salem'
 'Dehradun' 'Valsad' 'Haldwani' 'Dharwad' 'Surat' 'Indore' 'Karnal'
 'Panchkula' 'Mysore' 'Rohtak' 'Ambala Cantt' 'Samastipur' 'Panvel'
 'Purnea' 'Bhubaneswa

Everything looks correct, with no discernable typos save for `FuelType`, in which there does not appear to be a distinction between `CNG` and `CNG + CNG`, hence the two entries will be consolidated:

In [10]:
df["Fuel Type"] = df["Fuel Type"].replace("CNG + CNG", "CNG")
df["Fuel Type"].unique()

array(['Petrol', 'Diesel', 'CNG', 'LPG', 'Hybrid', 'Petrol + CNG'],
      dtype=object)

Lastly, the `Year`, `Transmission`, and `Owner` fields may be made generally numerically rather than categorical, producing `Age`, `Manual`, and `Owners` fields, respectively. Dropping the categorical fields, we produce:

In [11]:
df["Age"] = 2026 - df["Year"]
df.drop("Year", axis=1, inplace=True)

df["Manual"] = df["Transmission"].map({"Automatic": 1, "Manual": 0})
df.drop("Transmission", axis=1, inplace=True)

df["Owners"] = df["Owner"].map({"UnRegistered Car": 0, "First": 1, "Second": 2, "Third": 3})
df.drop("Owner", axis=1, inplace=True)

df

,Make,Price,Kilometer,Fuel Type,Location,Color,Seller Type,Engine,Max Power,Max Torque,Drivetrain,Length,Width,Height,Seating Capacity,Fuel Tank Capacity,Age,Manual,Owners
0,Honda,505000,87150,Petrol,Pune,Grey,Corporate,1198.0,87.0,109.0000,FWD,3990.0,1680.0,1505.0,5.0,35.0,9,0,1
1,Maruti Suzuki,450000,75000,Diesel,Ludhiana,White,Individual,1248.0,74.0,190.0000,FWD,3995.0,1695.0,1555.0,5.0,42.0,12,0,2
2,Hyundai,220000,67000,Petrol,Lucknow,Maroon,Individual,1197.0,79.0,112.7619,FWD,3585.0,1595.0,1550.0,5.0,35.0,15,0,1
3,Toyota,799000,37500,Petrol,Mangalore,Red,Individual,1197.0,82.0,113.0000,FWD,3995.0,1745.0,1510.0,5.0,37.0,7,0,1
4,Toyota,1950000,69000,Diesel,Mumbai,Grey,Individual,2393.0,148.0,343.0000,RWD,4735.0,1830.0,1795.0,7.0,55.0,8,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2053,Maruti Suzuki,245000,79000,Petrol,Faridabad,White,Individual,1197.0,85.0,113.0000,FWD,3775.0,1680.0,1620.0,5.0,43.0,12,0,2
2054,Mahindra,850000,90300,Diesel,Surat,White,Individual,2179.0,138.0,330.0000,FWD,4585.0,1890.0,1785.0,7.0,70.0,10,0,1
2055,Hyundai,275000,83000,Petrol,Ahmedabad,White,Individual,814.0,55.0,75.0000,FWD,3495.0,1550.0,1500.0,5.0,32.0,12,0,2
2056,Ford,240000,73000,Petrol,Thane,Silver,Individual,1196.0,70.0,102.0000,FWD,3795.0,1680.0,1427.0,5.0,45.0,13,0,1


Categorical data needs to now be encoded for our linear model to meaningfully interpret. For this purpose, we use standard one hot encoding.

In [12]:
from sklearn.preprocessing import OneHotEncoder

cols = ['Fuel Type','Drivetrain','Seller Type','Color','Make','Location']
ohe = OneHotEncoder(drop= 'first', handle_unknown='error', sparse_output=False).set_output(transform='pandas')
ohetransform = ohe.fit_transform(df[cols])
df = pd.concat([df , ohetransform], axis=1).drop(columns = cols)

df.head(10)

,Price,Kilometer,Engine,Max Power,Max Torque,Length,Width,Height,Seating Capacity,Fuel Tank Capacity,Age,Manual,Owners,Fuel Type_Diesel,Fuel Type_Hybrid,Fuel Type_LPG,Fuel Type_Petrol,Fuel Type_Petrol + CNG,Drivetrain_FWD,Drivetrain_RWD,Seller Type_Corporate,Seller Type_Individual,Color_Black,Color_Blue,Color_Bronze,Color_Brown,Color_Gold,Color_Green,Color_Grey,Color_Maroon,Color_Orange,Color_Others,Color_Purple,Color_Red,Color_Silver,Color_White,Color_Yellow,Make_BMW,Make_Chevrolet,Make_Datsun,...,Location_Kheda,Location_Kolkata,Location_Kollam,Location_Kota,Location_Lucknow,Location_Ludhiana,Location_Mangalore,Location_Meerut,Location_Mirzapur,Location_Mohali,Location_Mumbai,Location_Muzaffurpur,Location_Mysore,Location_Nagpur,Location_Nashik,Location_Navi Mumbai,Location_Noida,Location_Panchkula,Location_Panvel,Location_Patna,Location_Pimpri-Chinchwad,Location_Pune,Location_Purnea,Location_Raipur,Location_Ranchi,Location_Ranga Reddy,Location_Rohtak,Location_Roorkee,Location_Rudrapur,Location_Salem,Location_Samastipur,Location_Surat,Location_Thane,Location_Udupi,Location_Vadodara,Location_Valsad,Location_Varanasi,Location_Warangal,Location_Yamunanagar,Location_Zirakpur
0,505000,87150,1198.0,87.0,109.0000,3990.0,1680.0,1505.0,5.0,35.0,9,0,1,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,450000,75000,1248.0,74.0,190.0000,3995.0,1695.0,1555.0,5.0,42.0,12,0,2,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,220000,67000,1197.0,79.0,112.7619,3585.0,1595.0,1550.0,5.0,35.0,15,0,1,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,799000,37500,1197.0,82.0,113.0000,3995.0,1745.0,1510.0,5.0,37.0,7,0,1,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1950000,69000,2393.0,148.0,343.0000,4735.0,1830.0,1795.0,7.0,55.0,8,0,1,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,675000,73315,1373.0,91.0,130.0000,4490.0,1730.0,1485.0,5.0,43.0,9,0,1,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,2650000,75000,1995.0,188.0,400.0000,4439.0,1821.0,1612.0,5.0,51.0,9,1,2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,1390000,56000,1798.0,177.0,250.0000,4670.0,1814.0,1476.0,5.0,50.0,9,1,1,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,575000,85000,1461.0,84.0,200.0000,4331.0,1822.0,

As alluded to previously, it is also worth testing if the removal of `Locations` and/or `Color` would result in a better model

In [13]:
df_no_loc = df.drop(df.filter(regex='Location_.+').columns, axis=1)
df_no_col = df.drop(df.filter(regex='Color_.+').columns, axis=1)
df_no_loc_no_col = df.drop(df.filter(regex='(Location|Color)_.+').columns, axis=1)

df_no_loc_no_col

,Price,Kilometer,Engine,Max Power,Max Torque,Length,Width,Height,Seating Capacity,Fuel Tank Capacity,Age,Manual,Owners,Fuel Type_Diesel,Fuel Type_Hybrid,Fuel Type_LPG,Fuel Type_Petrol,Fuel Type_Petrol + CNG,Drivetrain_FWD,Drivetrain_RWD,Seller Type_Corporate,Seller Type_Individual,Make_BMW,Make_Chevrolet,Make_Datsun,Make_Ferrari,Make_Fiat,Make_Ford,Make_Honda,Make_Hyundai,Make_Isuzu,Make_Jaguar,Make_Jeep,Make_Kia,Make_Land Rover,Make_Lexus,Make_MG,Make_MINI,Make_Mahindra,Make_Maruti Suzuki,Make_Maserati,Make_Mercedes-Benz,Make_Mitsubishi,Make_Nissan,Make_Porsche,Make_Renault,Make_Rolls-Royce,Make_Skoda,Make_Ssangyong,Make_Tata,Make_Toyota,Make_Volkswagen,Make_Volvo
0,505000,87150,1198.0,87.0,109.0000,3990.0,1680.0,1505.0,5.0,35.0,9,0,1,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,450000,75000,1248.0,74.0,190.0000,3995.0,1695.0,1555.0,5.0,42.0,12,0,2,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,220000,67000,1197.0,79.0,112.7619,3585.0,1595.0,1550.0,5.0,35.0,15,0,1,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,799000,37500,1197.0,82.0,113.0000,3995.0,1745.0,1510.0,5.0,37.0,7,0,1,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,1950000,69000,2393.0,148.0,343.0000,4735.0,1830.0,1795.0,7.0,55.0,8,0,1,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2053,245000,79000,1197.0,85.0,113.0000,3775.0,1680.0,1620.0,5.0,43.0,12,0,2,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2054,850000,90300,2179.0,138.0,330.0000,4585.0,1890.0,1785.0,7.0,70.0,10,0,1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2055,275000,83000,814.0,55.0,75.0000,3495.0,1550.0,1500.0,5.0,32.0,12,0,2,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2056,240000,73000,1196.0,70.0,102.0000,3795.0,1680.0,1427.0,5.0,45.0,13,0,1,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Continuous data similarly needs to be scaled. In order to affect training bias, we must do so after splitting the data. Due to the several above data sets from which to choose, we create a function for this purpose:

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
SCALE_COLS = ["Kilometer", "Engine", "Max Power", "Max Torque",
              "Length", "Width", "Height", "Seating Capacity", "Fuel Tank Capacity", "Age", "Owners"]

def prepare_data(dat):
    X = dat.drop("Price", axis=1)
    Y = dat["Price"]
    
    X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=0xDEADBEEF)
    
    scaler = StandardScaler()
    X_train[SCALE_COLS] = scaler.fit_transform(X_train[SCALE_COLS])
    X_test[SCALE_COLS] = scaler.transform(X_test[SCALE_COLS])
    
    return X_train, X_test, y_train, y_test

## Part 2: Model Creation and Evaluation

It is unknown which model will fare the best. In order to process the several above data sets expeditiously, we again create a helper function for model creation and evaluation:

In [15]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

def train_and_test_model(dat, model):
    X_train, X_test, y_train, y_test = prepare_data(dat)
    
    model.fit(X_train, y_train)
    y_hat = model.predict(X_test)
    
    print(f"R^2: {r2_score(y_test, y_hat):0.3f}")
    print(f"MAE: {mean_absolute_error(y_test, y_hat):0.3f}")
    print(f"MAPE: {100*mean_absolute_percentage_error(y_test, y_hat):0.3f}%\n")

Next, we examine several potential models and parameters:

In [16]:
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

NAMES = ["df", "df_no_loc", "df_no_col", "df_no_loc_no_col"]
ITER = [1, 2, 2, 3]

# Begin with default settings
for i, x in enumerate([df, df_no_loc, df_no_col, df_no_loc_no_col]):
    # High tolerance to prototype solution
    for y in [LinearRegression(), Lasso(tol=1e-3), Ridge(tol=1e-3)]:
        for j in range(1,ITER[i]+1):
            print(f"{NAMES[i]}: {y.__class__.__name__} (N={j})")
            train_and_test_model(x, make_pipeline(PolynomialFeatures(degree=j), y))

df: LinearRegression (N=1)
R^2: 0.690
MAE: 761732.709
MAPE: 79.949%

df: Lasso (N=1)
R^2: 0.690
MAE: 761666.243
MAPE: 79.926%

df: Ridge (N=1)
R^2: 0.665
MAE: 797713.684
MAPE: 79.837%

df_no_loc: LinearRegression (N=1)
R^2: 0.698
MAE: 730211.744
MAPE: 72.218%

df_no_loc: LinearRegression (N=2)
R^2: -15.341
MAE: 2460238.485
MAPE: 122.036%

df_no_loc: Lasso (N=1)
R^2: 0.698
MAE: 730182.985
MAPE: 72.211%

df_no_loc: Lasso (N=2)


c:\Users\burke\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.914e+13, tolerance: 8.626e+12
  model = cd_fast.enet_coordinate_descent(


R^2: 0.783
MAE: 530861.993
MAPE: 40.770%

df_no_loc: Ridge (N=1)
R^2: 0.673
MAE: 769450.717
MAPE: 74.971%

df_no_loc: Ridge (N=2)
R^2: 0.904
MAE: 371094.051
MAPE: 29.986%

df_no_col: LinearRegression (N=1)
R^2: -951141220663589863424.000
MAE: 4126913512762358.500
MAPE: 1069898252167.039%

df_no_col: LinearRegression (N=2)
R^2: -3535.276
MAE: 34863242.071
MAPE: 2666.599%

df_no_col: Lasso (N=1)
R^2: 0.695
MAE: 744054.768
MAPE: 77.857%

df_no_col: Lasso (N=2)


c:\Users\burke\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.286e+13, tolerance: 8.626e+12
  model = cd_fast.enet_coordinate_descent(


R^2: 0.781
MAE: 671729.151
MAPE: 61.839%

df_no_col: Ridge (N=1)
R^2: 0.669
MAE: 788173.810
MAPE: 78.225%

df_no_col: Ridge (N=2)
R^2: 0.893
MAE: 407057.159
MAPE: 36.833%

df_no_loc_no_col: LinearRegression (N=1)
R^2: 0.702
MAE: 713779.584
MAPE: 70.062%

df_no_loc_no_col: LinearRegression (N=2)
R^2: -360959186609224.938
MAE: 6145310843370.322
MAPE: 1082019182.485%

df_no_loc_no_col: LinearRegression (N=3)
R^2: -93531.163
MAE: 122080394.839
MAPE: 5582.343%

df_no_loc_no_col: Lasso (N=1)
R^2: 0.702
MAE: 713767.138
MAPE: 70.058%

df_no_loc_no_col: Lasso (N=2)


c:\Users\burke\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.304e+13, tolerance: 8.626e+12
  model = cd_fast.enet_coordinate_descent(


R^2: 0.775
MAE: 427197.201
MAPE: 28.904%

df_no_loc_no_col: Lasso (N=3)
R^2: 0.816
MAE: 459683.919
MAPE: 28.395%

df_no_loc_no_col: Ridge (N=1)
R^2: 0.676
MAE: 759864.695
MAPE: 73.446%

df_no_loc_no_col: Ridge (N=2)
R^2: 0.909
MAE: 328052.070
MAPE: 25.549%

df_no_loc_no_col: Ridge (N=3)
R^2: 0.886
MAE: 342526.774
MAPE: 21.091%



## Part 3: Tuning

From the above pilot test, we indeed find that `df_no_loc_no_col` (that is, the original data with location and color excluded), is generally superior in terms of MAPE, especially with a higher degree polynomial kernel. Given this result, we therefore attempt to tune the corresponding `alpha` parameter for high degree:

In [17]:
# Default tolerance for tuning
for N in range(2,5):
    # Several orders of magnitude of alpha for comparison
    for a in [0.01, 0.1, 1.0, 10.0, 100.0]:
        print(f"df_no_loc_no_col: Ridge (N={N}, a={a})")
        train_and_test_model(df_no_loc_no_col, make_pipeline(PolynomialFeatures(degree=N), Ridge(alpha=a)))

df_no_loc_no_col: Ridge (N=2, a=0.01)
R^2: 0.726
MAE: 432807.494
MAPE: 30.168%

df_no_loc_no_col: Ridge (N=2, a=0.1)
R^2: 0.851
MAE: 359262.930
MAPE: 27.632%

df_no_loc_no_col: Ridge (N=2, a=1.0)
R^2: 0.909
MAE: 328052.070
MAPE: 25.549%

df_no_loc_no_col: Ridge (N=2, a=10.0)
R^2: 0.913
MAE: 339999.200
MAPE: 26.563%

df_no_loc_no_col: Ridge (N=2, a=100.0)
R^2: 0.865
MAE: 408531.180
MAPE: 33.242%

df_no_loc_no_col: Ridge (N=3, a=0.01)
R^2: 0.150
MAE: 747121.618
MAPE: 44.371%

df_no_loc_no_col: Ridge (N=3, a=0.1)
R^2: 0.681
MAE: 483971.542
MAPE: 28.312%

df_no_loc_no_col: Ridge (N=3, a=1.0)
R^2: 0.886
MAE: 342526.774
MAPE: 21.091%

df_no_loc_no_col: Ridge (N=3, a=10.0)
R^2: 0.920
MAE: 301274.694
MAPE: 18.357%

df_no_loc_no_col: Ridge (N=3, a=100.0)
R^2: 0.934
MAE: 285908.033
MAPE: 17.076%

df_no_loc_no_col: Ridge (N=4, a=0.01)
R^2: -13.092
MAE: 1956088.362
MAPE: 113.173%

df_no_loc_no_col: Ridge (N=4, a=0.1)
R^2: -0.415
MAE: 978982.838
MAPE: 56.128%

df_no_loc_no_col: Ridge (N=4, a=1.0)
R

From this result, we find the lowest MAPE with highest R^2 value near (N=3, a=100). This simultaneously captures both the overall trends in the data set along with minimizing the percentage error per vehicle. Performing some final fine tuning, we find:

In [18]:
# Fine alpha tuning
for a in [10.0, 25.0, 50.0, 100.0, 250.0, 500.0, 1000.0]:
    print(f"df_no_loc_no_col: Ridge (N=3, a={a})")
    train_and_test_model(df_no_loc_no_col, make_pipeline(PolynomialFeatures(degree=3), Ridge(alpha=a)))

df_no_loc_no_col: Ridge (N=3, a=10.0)
R^2: 0.920
MAE: 301274.694
MAPE: 18.357%

df_no_loc_no_col: Ridge (N=3, a=25.0)
R^2: 0.929
MAE: 286318.407
MAPE: 17.337%

df_no_loc_no_col: Ridge (N=3, a=50.0)
R^2: 0.933
MAE: 283277.506
MAPE: 16.963%

df_no_loc_no_col: Ridge (N=3, a=100.0)
R^2: 0.934
MAE: 285908.033
MAPE: 17.076%

df_no_loc_no_col: Ridge (N=3, a=250.0)
R^2: 0.930
MAE: 297088.918
MAPE: 18.163%

df_no_loc_no_col: Ridge (N=3, a=500.0)
R^2: 0.923
MAE: 310514.041
MAPE: 19.435%

df_no_loc_no_col: Ridge (N=3, a=1000.0)
R^2: 0.913
MAE: 329991.331
MAPE: 21.460%



In [19]:
# Finest alpha tuning
for a in range(50, 150, 10):
    print(f"df_no_loc_no_col: Ridge (N=3, a={a})")
    train_and_test_model(df_no_loc_no_col, make_pipeline(PolynomialFeatures(degree=3), Ridge(alpha=a)))

df_no_loc_no_col: Ridge (N=3, a=50)
R^2: 0.933
MAE: 283277.506
MAPE: 16.963%

df_no_loc_no_col: Ridge (N=3, a=60)
R^2: 0.934
MAE: 283564.859
MAPE: 16.945%

df_no_loc_no_col: Ridge (N=3, a=70)
R^2: 0.934
MAE: 284117.494
MAPE: 16.962%

df_no_loc_no_col: Ridge (N=3, a=80)
R^2: 0.934
MAE: 284740.662
MAPE: 17.000%

df_no_loc_no_col: Ridge (N=3, a=90)
R^2: 0.934
MAE: 285350.618
MAPE: 17.037%

df_no_loc_no_col: Ridge (N=3, a=100)
R^2: 0.934
MAE: 285908.033
MAPE: 17.076%

df_no_loc_no_col: Ridge (N=3, a=110)
R^2: 0.934
MAE: 286481.867
MAPE: 17.128%

df_no_loc_no_col: Ridge (N=3, a=120)
R^2: 0.933
MAE: 287139.418
MAPE: 17.187%

df_no_loc_no_col: Ridge (N=3, a=130)
R^2: 0.933
MAE: 287843.329
MAPE: 17.277%

df_no_loc_no_col: Ridge (N=3, a=140)
R^2: 0.933
MAE: 288506.302
MAPE: 17.369%



## Part 4: Final Model and Verification

Ergo, we find a minimized model near (N=3, a=60). For practical purposes, we stop here, as increased granularity is unlikely to meaningfully improve our model. Thus, we construct our final model manually for analysis:

In [20]:
X_train, X_test, y_train, y_test = prepare_data(df_no_loc_no_col)

model = make_pipeline(PolynomialFeatures(degree=3), Ridge(alpha=60, tol=1e-6, max_iter=int(1e5)))
    
model.fit(X_train, y_train)
y_hat = model.predict(X_test)
print("Final trained model evaluation")
print(f"R^2: {r2_score(y_test, y_hat):0.3f}")
print(f"MAE: {mean_absolute_error(y_test, y_hat):0.3f}")
print(f"MAPE: {100*mean_absolute_percentage_error(y_test, y_hat):0.3f}%")

# Median as measure of central error
e = y_hat - y_test
APE = abs(e/y_test)
print(f"MedAPE: {100*APE.median():0.3f}%")
print(f"Percentile of 15% error: {100*(sum(APE<=0.15)/len(APE)):0.3f} \n")

Final trained model evaluation
R^2: 0.934
MAE: 283564.859
MAPE: 16.945%
MedAPE: 10.946%
Percentile of 15% error: 63.733 



Hence, we show that our model captures 93.4% of the variance of our data, making it extensible, while accurately predicting the fair market rate of 63.7% of cars within 15%. This particular benchmark has been chosen due to the problem outline citing 15% as the approximate limit of user tolerance.